# AG News: LSTM vs Transformer Encoder

최종 제출용 notebook 초안입니다. `data_pipeline.py`와 `models.py`를 import해서 실행합니다.

- 데이터 소스: HuggingFace `ag_news` 하나만 씁니다 (TorchText는 tokenizer와 vocab 유틸리티로만).
- 환경: `conda activate agnews-dl` (`setup_env.ps1`로 만든 환경).

전체 흐름은 파이프라인 로드, 데이터 문서화, 배치 확인, baseline smoke test, LSTM과 Transformer 학습, 평가와 그림, 표 순서입니다.

## 1. 환경 설정과 파이프라인 로드

In [1]:
import torch
import torch.nn as nn
from data_pipeline import DataConfig, build_pipeline, describe_pipeline, set_seed, to_device
from models import (
    AverageEmbeddingClassifier,
    LSTMClassifier,
    TransformerEncoderClassifier,
    count_parameters,
)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

c:\Users\song\miniconda3\envs\agnews-dl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu


c:\Users\song\miniconda3\envs\agnews-dl\Lib\site-packages\torchtext\data\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
c:\Users\song\miniconda3\envs\agnews-dl\Lib\site-packages\torchtext\vocab\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
c:\Users\song\miniconda3\envs\agnews-dl\Lib\site-packages\torchtext\utils.py:4: UserWarning: 
/!\ IMPORTANT WARNI

In [2]:
cfg = DataConfig()
bundle = build_pipeline(cfg)
describe_pipeline(bundle)  # dataset documentation 

AG News pipeline summary
dataset source    : HuggingFace 'ag_news' (single source)
tokenizer         : torchtext basic_english (tokenizer/vocab utility only)
seed              : 42   |  val_ratio: 0.1  |  train_fraction: 1.0
max_len           : 128  |  batch_size: 64
padding strategy  : batch-wise dynamic padding (capped at max_len=128)
vocab size (built): 20000 (cap 20000)  |  pad=0 unk=1
num classes       : 4
label mapping     : {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

-- split sizes --
  train: 108,000
  val  : 12,000
  test : 7,600

-- class distribution (count per label index) --
  train: {0: 26991, 1: 26966, 2: 27100, 3: 26943}
  val  : {0: 3009, 1: 3034, 2: 2900, 3: 3057}
  test : {0: 1900, 1: 1900, 2: 1900, 3: 1900}

-- sequence length (pre-truncation tokens) & truncation rate --
  train: mean=43.27  p95=65   max=207   trunc_rate@128=0.14%
  val  : mean=43.35  p95=65   max=172   trunc_rate@128=0.17%
  test : mean=43.04  p95=63   max=161   trunc_rate@128=0.18%

-

## 2. 배치 shape 확인

In [3]:
batch = next(iter(bundle.train_loader))
for k, v in batch.items():
    shape = tuple(v.shape) if torch.is_tensor(v) else f"list[{len(v)}]"
    dtype = v.dtype if torch.is_tensor(v) else type(v[0]).__name__
    print(f"{k:14s} {str(shape):14s} {dtype}")

input_ids      (64, 79)       torch.int64
lengths        (64,)          torch.int64
labels         (64,)          torch.int64
texts          list[64]       str
indices        list[64]       int
orig_lengths   (64,)          torch.int64
truncated      (64,)          torch.bool


## 3. Baseline smoke test (AverageEmbeddingClassifier)

파이프라인이 batch에서 model을 거쳐 loss까지 잘 연결되는지 확인하는 용도입니다. 필수 실험 모델은 아닙니다.

In [ ]:
set_seed(42)
baseline = AverageEmbeddingClassifier(
    bundle.vocab_size, bundle.num_classes, bundle.pad_idx, embed_dim=128
).to(device)

b = to_device(next(iter(bundle.train_loader)), device)
logits = baseline(b["input_ids"], b["lengths"])
loss = nn.CrossEntropyLoss()(logits, b["labels"])
print("logits:", tuple(logits.shape), "| loss:", round(loss.item(), 4), "| params:", count_parameters(baseline))

## 4. 공유 학습과 평가 루프

LSTM과 Transformer가 똑같이 쓰는 학습 루프입니다.
- 학습은 `model.train()`, 평가는 `model.eval()`과 `torch.no_grad()`로 합니다.
- 모델 선택은 검증셋(val) 기준으로만 합니다 (test는 쓰지 않음). 점수는 macro-F1으로 봅니다.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    for batch in loader:
        batch = to_device(batch, device)
        optimizer.zero_grad()
        logits = model(batch["input_ids"], batch["lengths"])
        loss = criterion(logits, batch["labels"])
        loss.backward()
        optimizer.step()
        total += loss.item() * batch["labels"].size(0)
    return total / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total = 0.0
    y_true, y_pred = [], []
    for batch in loader:
        batch = to_device(batch, device)
        logits = model(batch["input_ids"], batch["lengths"])
        total += criterion(logits, batch["labels"]).item() * batch["labels"].size(0)
        y_true.extend(batch["labels"].cpu().tolist())
        y_pred.extend(logits.argmax(1).cpu().tolist())
    avg = total / len(loader.dataset)
    return avg, accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro"), y_true, y_pred


def run_training(build_model, bundle, epochs=8, lr=1e-3, device=device):
    set_seed(42)
    model = build_model().to(device)
    print("trainable params:", count_parameters(model))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}
    best_f1, best_state = -1.0, None
    for ep in range(1, epochs + 1):
        tr = train_one_epoch(model, bundle.train_loader, opt, crit, device)
        vl, acc, f1, _, _ = evaluate(model, bundle.val_loader, crit, device)
        history["train_loss"].append(tr)
        history["val_loss"].append(vl)
        history["val_acc"].append(acc)
        history["val_f1"].append(f1)
        print(f"epoch {ep}: train_loss={tr:.4f} val_loss={vl:.4f} val_acc={acc:.4f} val_f1={f1:.4f}")
        if f1 > best_f1:  # pick the best epoch on validation only
            best_f1 = f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, history

## 5. LSTM 학습

In [ ]:
lstm, lstm_hist = run_training(
    lambda: LSTMClassifier(
        bundle.vocab_size, bundle.num_classes, bundle.pad_idx,
        embed_dim=128, hidden_size=256, num_layers=2, dropout=0.3,
    ),
    bundle, epochs=8,
)

## 6. Transformer Encoder 학습

In [ ]:
tr_model, tr_hist = run_training(
    lambda: TransformerEncoderClassifier(
        bundle.vocab_size, bundle.num_classes, bundle.pad_idx,
        embed_dim=128, nhead=4, num_layers=2, dim_feedforward=256,
        dropout=0.3, max_len=cfg.max_len,
    ),
    bundle, epochs=8,
)

## 7. 최종 test 평가, 그림과 표, ablation, 실패 분석

- 두 모델의 best(검증 기준)를 고른 뒤 `bundle.test_loader`로 한 번만 평가합니다.
- 손실 곡선(train과 val), 혼동 행렬(`confusion_matrix`와 `bundle.class_names`), accuracy와 macro-F1 표를 만듭니다.
- 파라미터 수 표는 두 모델 모두 `count_parameters`로 구합니다.
- 필수 embedding dim ablation: `DataConfig`는 그대로 두고 모델의 `embed_dim`만 64, 128, 256으로 바꿔 반복합니다.
- 학습 곡선: `build_pipeline(DataConfig(train_fraction=0.25/0.5/1.0))`로 bundle을 다시 만듭니다.
- 실패 분석: val이나 test 배치의 `texts`, `indices`, `truncated`로 오분류 예시를 뽑습니다 (공유, LSTM만, Transformer만 각각 5건 이상).